In [19]:

# Importing Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler #converts text to numbers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score #main metric for evaluation
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [20]:
# importing training data
train = pd.read_csv('train.csv')
train.head()
# importing testing data
test = pd.read_csv('test.csv') 
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [21]:
# basic prep
# drop id
train = train.drop("id", axis=1)
test_ids = test["id"]
test = test.drop("id", axis=1)

# target
y = train["Irrigation_Need"]
X = train.drop("Irrigation_Need", axis=1)

In [22]:
# encoding and aligning
X = pd.get_dummies(X)
test = pd.get_dummies(test)


X, test = X.align(test, join="left", axis=1, fill_value=0)

# train and validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [23]:
#feature engineering
# interaction
X_train["feature_interaction"] = X_train.iloc[:, 0] * X_train.iloc[:, 1]
X_val["feature_interaction"] = X_val.iloc[:, 0] * X_val.iloc[:, 1]

# log transform
X_train["log_feature"] = np.log1p(X_train.iloc[:, 2])
X_val["log_feature"] = np.log1p(X_val.iloc[:, 2])

# binning
X_train["feature_bin"] = pd.cut(X_train.iloc[:, 3], bins=5, labels=False)
X_val["feature_bin"] = pd.cut(X_val.iloc[:, 3], bins=5, labels=False)

# ratio feature
X_train["ratio_feature"] = X_train.iloc[:, 0] / (X_train.iloc[:, 2] + 1)
X_val["ratio_feature"] = X_val.iloc[:, 0] / (X_val.iloc[:, 2] + 1)

# frequency encoding (example using first column)
freq = X_train.iloc[:, 0].value_counts()
X_train["freq_feature"] = X_train.iloc[:, 0].map(freq)
X_val["freq_feature"] = X_val.iloc[:, 0].map(freq)

In [24]:
from sklearn.model_selection import GridSearchCV

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

log_model = LogisticRegression(max_iter=1000, random_state=42)

param_grid_log = {
    "C": [0.1, 0.5, 1.0],
    "class_weight": [None, "balanced"]
}

grid_log = GridSearchCV(
    log_model,
    param_grid_log,
    cv=3,
    scoring="balanced_accuracy"
)

grid_log.fit(X_train_scaled, y_train)

print("Best Logistic Params:", grid_log.best_params_)

best_log = grid_log.best_estimator_

log_pred = best_log.predict(X_val_scaled)

print("Logistic Regression Score:",
      balanced_accuracy_score(y_val, log_pred))

Best Logistic Params: {'C': 1.0, 'class_weight': 'balanced'}
Logistic Regression Score: 0.8512673734340299


In [25]:
lgbm = LGBMClassifier(random_state=42)

param_grid_lgbm = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1]
}

grid_lgbm = GridSearchCV(
    lgbm,
    param_grid_lgbm,
    cv=3,
    scoring="balanced_accuracy"
)

grid_lgbm.fit(X_train, y_train)

print("Best LGBM Params:", grid_lgbm.best_params_)

best_lgbm = grid_lgbm.best_estimator_

lgbm_pred = best_lgbm.predict(X_val)

print("LightGBM Score:",
      balanced_accuracy_score(y_val, lgbm_pred))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005468 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3609
[LightGBM] [Info] Number of data points in the train set: 336000, number of used features: 48
[LightGBM] [Info] Start training from score -3.400751
[LightGBM] [Info] Start training from score -0.532442
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

In [27]:
#ensemble
log_probs = best_log.predict_proba(X_val_scaled)
lgbm_probs = best_lgbm.predict_proba(X_val)

avg_probs = (log_probs + lgbm_probs) / 2

ensemble_pred = best_log.classes_[np.argmax(avg_probs, axis=1)]

print("Ensemble Score:",
      balanced_accuracy_score(y_val, ensemble_pred))

Ensemble Score: 0.961758920514749


In [28]:
# feature importance
feat_imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_lgbm.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 Important Features:")
print(feat_imp.head(15))


Top 15 Important Features:
                         feature  importance
6                    Rainfall_mm        3040
4                  Temperature_C        2114
1                  Soil_Moisture        1838
8                 Wind_Speed_kmh        1809
36              Mulching_Used_No        1478
10        Previous_Irrigation_mm         977
5                       Humidity         930
22     Crop_Growth_Stage_Harvest         785
23      Crop_Growth_Stage_Sowing         639
7                 Sunlight_Hours         509
3        Electrical_Conductivity         470
9             Field_Area_hectare         457
24  Crop_Growth_Stage_Vegetative         405
21   Crop_Growth_Stage_Flowering         378
0                        Soil_pH         368


## Kaggle Submissions


In [ ]:
# applying same feature engineering to test FIRST

test["feature_interaction"] = test.iloc[:, 0] * test.iloc[:, 1]
test["log_feature"] = np.log1p(test.iloc[:, 2])
test["feature_bin"] = pd.cut(test.iloc[:, 3], bins=5, labels=False)
test["ratio_feature"] = test.iloc[:, 0] / (test.iloc[:, 2] + 1)

freq = X_train.iloc[:, 0].value_counts()
test["freq_feature"] = test.iloc[:, 0].map(freq)

# cleaning up missing values
test["feature_bin"] = test["feature_bin"].fillna(0).astype(int)
test["freq_feature"] = test["freq_feature"].fillna(0)

# match training columns exactly
test = test.reindex(columns=X_train.columns, fill_value=0)

# scaling
test_scaled = scaler.transform(test)

In [32]:
# logistic
test_scaled = scaler.transform(test)
log_test_probs = best_log.predict_proba(test_scaled)
log_preds = best_log.classes_[np.argmax(log_test_probs, axis=1)]

submission_log = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": log_preds
})

submission_log.to_csv("logistic_submission.csv", index=False)

In [33]:
#lightgbm
lgbm_test_preds = best_lgbm.predict(test)

submission_lgbm = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": lgbm_test_preds
})

submission_lgbm.to_csv("lgbm_submission.csv", index=False)

In [34]:
# ensemble
lgbm_test_probs = best_lgbm.predict_proba(test)

avg_test_probs = (log_test_probs + lgbm_test_probs) / 2

ensemble_test_preds = best_log.classes_[np.argmax(avg_test_probs, axis=1)]

submission_ens = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": ensemble_test_preds
})

submission_ens.to_csv("ensemble_submission.csv", index=False)